# 공공서비스 공급 데이터 전처리

## 목적

부산 지역의 생활수요와 공공서비스 공급 간 불균형을 분석하기 위해
고령층 및 아동 관련 공공서비스 시설 데이터를 수집·검증·전처리한다.

### 활용 데이터

- 경로당 현황
- 노인교실 현황
- 아동복지 및 시설 정보

### 주요 작업

1. 원본 데이터 구조 및 품질 검증
2. 시설 유형 및 분석 대상 선정
3. 주소·좌표 등 공간정보 정제
4. 행정동 단위 시설 공급량 산출
5. 생활수요 데이터와 결합 가능한 형태로 가공

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import requests
from google.colab import userdata

SERVICE_KEY = userdata.get("BUSAN_API_KEY")

url = "https://apis.data.go.kr/6260000/ChildWelfareService2/ChildWelfareFacilityInfo2"

params = {
    "serviceKey": SERVICE_KEY,
    "gugun": "해운대구",
    "numOfRows": 10,
    "pageNo": 1,
    "resultType": "json"
}

response = requests.get(url, params=params, timeout=30)

print("상태코드:", response.status_code)
print(response.text[:3000])

## 아동복지시설 API 연결 확인

부산광역시 아동 복지 및 시설 정보 OpenAPI를 시험 호출한 결과
정상 응답(`NORMAL_CODE`)을 확인하였다.

API 전체 등록 건수는 284건이며,
시설명, 시설유형, 주소, 구·군, 정원, 현원, 위도·경도 등의 정보를 제공한다.

시험 요청에서 구·군 조건과 관계없이 다른 구의 시설이 함께 반환되는 것이 확인되어,
구·군별 반복 호출 대신 전체 데이터를 수집한 후 `gugun` 컬럼을 기준으로
지역을 구분하는 방식으로 처리한다.

또한 `reference_date`는 현재 응답에서 결측으로 제공되므로
분석 시 API 메타데이터의 제공시점과 실제 수집일을 별도로 기록한다.

In [ ]:
# 아동복지시설 전체 데이터를 한 번에 수집한다.
# API의 gugun 필터에 의존하지 않고 응답의 gugun 컬럼을 사용한다.
import pandas as pd

params = {
    "serviceKey": SERVICE_KEY,
    "numOfRows": 1000,
    "pageNo": 1,
    "resultType": "json"
}

response = requests.get(url, params=params, timeout=30)
data = response.json()

items = data["response"]["body"]["items"]["item"]
child_raw = pd.DataFrame(items)

print("수집 건수:", len(child_raw))
print("컬럼 수:", len(child_raw.columns))

print("\n시설유형별 건수")
print(child_raw["type_of_facility"].value_counts(dropna=False))

print("\n구·군별 건수")
print(child_raw["gugun"].value_counts(dropna=False).sort_index())

print("\n결측값")
print(child_raw.isna().sum())

In [ ]:
# 시설유형 값이 불명확한 19건의 실제 시설명과 주소를 확인한다.

check_types = [
    "법인",
    "개인",
    "사회적협동조합",
    "아동복지시설"
]

child_raw.loc[
    child_raw["type_of_facility"].isin(check_types),
    ["facility_nm", "type_of_facility", "addr", "gugun"]
].sort_values(["type_of_facility", "facility_nm"])

In [ ]:
# 원본 API 응답은 유지하고 분석용 데이터에서만
# 명백하게 잘못 분류된 지역아동센터 유형과 구·군 표기를 정정한다.

child_clean = child_raw.copy()

wrong_types = ["개인", "법인", "사회적협동조합"]

mask = (
    child_clean["type_of_facility"].isin(wrong_types)
    & child_clean["facility_nm"].str.contains("지역아동센터", na=False)
)

print("지역아동센터로 정정:", mask.sum())

child_clean.loc[mask, "type_of_facility"] = "지역아동센터"

child_clean["gugun"] = (
    child_clean["gugun"]
    .str.replace("부산광역시 ", "", regex=False)
    .str.strip()
)

print("\n정정 후 시설유형")
print(child_clean["type_of_facility"].value_counts())

print("\n구·군 수:", child_clean["gugun"].nunique())
print(child_clean["gugun"].value_counts().sort_index())

In [ ]:
# 일반 아동이 지역에서 이용할 수 있는 돌봄서비스를 분석 대상으로 선정한다.
# 보호·거주시설은 제외하고 지역아동센터와 다함께돌봄센터만 사용한다.

target_types = ["지역아동센터", "다함께돌봄센터"]

child_service = (
    child_clean[
        child_clean["type_of_facility"].isin(target_types)
    ]
    .copy()
    .reset_index(drop=True)
)

print("=== 분석 대상 ===")
print("전체:", len(child_service))
print(child_service["type_of_facility"].value_counts())

print("\n=== 지역 범위 ===")
print("구·군 수:", child_service["gugun"].nunique())
print(child_service["gugun"].value_counts().sort_index())

print("\n=== 핵심 변수 결측 ===")
check_cols = [
    "facility_nm", "type_of_facility",
    "addr", "gugun", "lat", "lng"
]
print(child_service[check_cols].isna().sum())

print("\n=== 중복 ===")
print("완전 중복:", child_service.duplicated().sum())
print(
    "시설명+주소 중복:",
    child_service.duplicated(
        subset=["facility_nm", "addr"]
    ).sum()
)

In [ ]:
# 중복 시설의 실제 내용을 확인한다.
# 삭제하기 전에 시설명·주소·유형·정원·현원·좌표를 비교한다.

dup_check = (
    child_service[
        child_service.duplicated(
            subset=["facility_nm", "addr"],
            keep=False
        )
    ]
    .sort_values(["facility_nm", "addr"])
)

display(
    dup_check[
        [
            "facility_nm",
            "type_of_facility",
            "addr",
            "gugun",
            "num_of_people",
            "present_num_of_pp",
            "lat",
            "lng"
        ]
    ]
)

In [ ]:
# 동일 시설명·주소로 중복 등록된 시설을 1개 시설로 처리한다.
# 원본 child_raw는 유지하고 분석용 child_service에서만 중복을 제거한다.

before = len(child_service)

child_service = (
    child_service
    .drop_duplicates(
        subset=["facility_nm", "addr"],
        keep="last"
    )
    .reset_index(drop=True)
)

after = len(child_service)

print("중복 제거 전:", before)
print("중복 제거 후:", after)
print("제거된 중복 행:", before - after)

print("\n시설유형별 건수")
print(child_service["type_of_facility"].value_counts())

print("\n구·군 수:", child_service["gugun"].nunique())

print("\n남은 시설명+주소 중복:",
      child_service.duplicated(
          subset=["facility_nm", "addr"]
      ).sum())

## 아동 돌봄서비스 공급 데이터 전처리 결과

부산광역시 아동 복지 및 시설 정보 OpenAPI를 통해
총 284건의 아동복지 관련 시설 데이터를 수집하였다.

### 시설유형 정제

시설유형을 확인한 결과 일부 지역아동센터 14건의
`type_of_facility` 값이 `개인`, `법인`, `사회적협동조합` 등
시설유형이 아닌 운영주체 형태로 입력되어 있었다.

해당 시설들은 시설명에 `지역아동센터`가 명시되어 있어
분석용 데이터에서 시설유형을 `지역아동센터`로 정정하였다.

또한 `부산광역시 동래구`, `부산광역시 수영구`로 기록된
구·군명을 각각 `동래구`, `수영구`로 표준화하였다.

### 분석 대상 선정

본 분석은 지역 일반 아동이 일상적으로 이용할 수 있는
지역 기반 돌봄서비스의 공급 수준을 측정하는 것을 목적으로 한다.

이에 다음 시설을 분석 대상으로 선정하였다.

- 지역아동센터
- 다함께돌봄센터

공동생활가정, 아동양육시설, 자립지원시설 등은
특정 보호·거주 목적의 시설이므로 지역 일반 아동의 생활수요와
직접 대응시키기 어렵다고 판단하여 공급지표에서 제외하였다.

### 중복시설 처리

분석 대상 추출 후 시설명과 주소를 기준으로 중복을 검증한 결과,
동구의 동일 지역아동센터들이 중복 등록된 사례가 확인되었다.

중복된 시설들은 시설명·주소·좌표가 동일하였으며,
일부 정원 또는 현원 값에 차이가 존재하였다.

본 분석에서는 시설의 정원·현원이 아닌 시설의 존재 여부와 개수를
공급지표로 활용하므로 `시설명 + 주소`가 동일한 시설은
하나의 시설로 처리하였다.

API 원본 데이터는 수정하지 않고 별도로 보존한다.

In [ ]:
# API 수집 원본과 최종 분석용 아동 돌봄시설 데이터를 각각 저장한다.

raw_path = (
    "/content/drive/MyDrive/부산_BigData_공모전/"
    "01_raw_data/공공서비스_공급/아동복지시설_API원본.csv"
)

processed_path = (
    "/content/drive/MyDrive/부산_BigData_공모전/"
    "02_processed_data/아동돌봄시설_분석용.csv"
)

child_raw.to_csv(
    raw_path,
    index=False,
    encoding="utf-8-sig"
)

child_service.to_csv(
    processed_path,
    index=False,
    encoding="utf-8-sig"
)

print("API 원본:", child_raw.shape)
print("분석용:", child_service.shape)
print("구·군 수:", child_service["gugun"].nunique())
print("저장 완료")

# 고령층 생활서비스 공급 데이터 전처리

고령층의 지역 생활서비스 공급 수준을 분석하기 위해
부산광역시 경로당 현황과 노인교실 현황 데이터를 활용한다.

- 경로당: 부산광역시 경로당 현황 Shapefile
- 노인교실: 부산광역시 노인교실 현황 CSV (2025.09.01)

먼저 각 데이터의 구조, 지역 범위, 결측 및 중복 여부를 검증한 후
분석에 사용할 시설 공급 데이터를 구성한다.

In [ ]:
import geopandas as gpd

senior_center_path = (
    "/content/drive/MyDrive/부산_BigData_공모전/"
    "01_raw_data/공공서비스_공급/부산광역시 경로당 현황.shp"
)

senior_center = gpd.read_file(senior_center_path)

print("데이터 크기:", senior_center.shape)
print("좌표계:", senior_center.crs)
print("\n컬럼:")
print(senior_center.columns.tolist())

display(senior_center.head())

In [ ]:
# 경로당 데이터의 결측, 중복, 지역 범위와 좌표 이상 여부를 확인한다.
# 실제 정제 전에 데이터 품질을 먼저 검증한다.

print("=== 기본 정보 ===")
print("전체 시설:", len(senior_center))
print("구·군 수:", senior_center["gugun"].nunique())

print("\n=== 구·군별 시설 수 ===")
print(senior_center["gugun"].value_counts().sort_index())

print("\n=== 결측값 ===")
print(
    senior_center[
        ["gugun", "name", "address", "lat", "lng"]
    ].isna().sum()
)

print("\n=== 빈 문자열 ===")
for col in ["gugun", "name", "address"]:
    print(
        col,
        senior_center[col].astype(str).str.strip().eq("").sum()
    )

print("\n=== 중복 ===")
print("완전 중복:", senior_center.duplicated().sum())
print(
    "시설명+주소 중복:",
    senior_center.duplicated(
        subset=["name", "address"]
    ).sum()
)

print("\n=== 좌표 범위 ===")
print("위도:", senior_center["lat"].min(), "~", senior_center["lat"].max())
print("경도:", senior_center["lng"].min(), "~", senior_center["lng"].max())

print("\n=== 좌표 결측/0 ===")
print("위도 0:", (senior_center["lat"] == 0).sum())
print("경도 0:", (senior_center["lng"] == 0).sum())

## 경로당 데이터 검증 결과

부산광역시 경로당 현황 Shapefile을 확인한 결과 총 2,374개 시설이 수록되어 있으며,
부산 16개 구·군이 모두 포함되어 있다.

시설명, 주소, 구·군, 위도·경도 등 핵심 변수에는 결측값이나 빈 문자열이 없었으며,
완전 중복 및 `시설명 + 주소` 기준 중복도 확인되지 않았다.

위도·경도 역시 결측 또는 0인 값이 없고 부산 지역 범위 내에 분포하여
별도의 시설 제거 없이 2,374개 전체 시설을 고령층 생활서비스 공급 분석에 활용한다.

In [ ]:
# 최신 노인교실 CSV를 불러와 데이터 구조와 기준시점을 확인한다.

senior_class_path = (
    "/content/drive/MyDrive/부산_BigData_공모전/"
    "01_raw_data/공공서비스_공급/"
    "부산광역시_노인교실 현황_20250901.csv"
)

senior_class = pd.read_csv(
    senior_class_path,
    encoding="cp949"
)

print("데이터 크기:", senior_class.shape)

print("\n컬럼:")
print(senior_class.columns.tolist())

display(senior_class.head())

In [ ]:
# 노인교실의 지역 범위, 결측, 중복 및 좌표 이상 여부를 확인한다.
# 정제 전에 원본 데이터의 품질을 먼저 검증한다.

print("=== 기본 정보 ===")
print("전체 시설:", len(senior_class))
print("구·군 수:", senior_class["구군명"].nunique())

print("\n=== 구·군별 시설 수 ===")
print(senior_class["구군명"].value_counts().sort_index())

print("\n=== 결측값 ===")
check_cols = [
    "구군명", "노인교실명", "도로명주소",
    "위도", "경도", "데이터기준일자"
]
print(senior_class[check_cols].isna().sum())

print("\n=== 빈 문자열 ===")
for col in ["구군명", "노인교실명", "도로명주소"]:
    print(
        col,
        senior_class[col].astype(str).str.strip().eq("").sum()
    )

print("\n=== 중복 ===")
print("완전 중복:", senior_class.duplicated().sum())
print(
    "시설명+주소 중복:",
    senior_class.duplicated(
        subset=["노인교실명", "도로명주소"]
    ).sum()
)

print("\n=== 좌표 범위 ===")
print("위도:", senior_class["위도"].min(), "~", senior_class["위도"].max())
print("경도:", senior_class["경도"].min(), "~", senior_class["경도"].max())

print("\n=== 기준일자 ===")
print(senior_class["데이터기준일자"].value_counts(dropna=False))

In [ ]:
# 부산 지역을 크게 벗어나는 좌표가 입력된 시설을 확인한다.
# 좌표를 임의 수정하지 않고 시설명과 주소를 먼저 검토한다.

coord_check = senior_class[
    ~(
        senior_class["위도"].between(34.9, 35.5)
        & senior_class["경도"].between(128.7, 129.4)
    )
]

print("부산 범위 밖 좌표:", len(coord_check))

display(
    coord_check[
        [
            "구군명",
            "노인교실명",
            "도로명주소",
            "경도",
            "위도",
            "데이터기준일자"
        ]
    ]
)

In [ ]:
# 구·군명 앞뒤 공백으로 동일 지역이 분리되어 있는지 확인한다.

for value in sorted(senior_class["구군명"].unique()):
    print(repr(value))

In [ ]:
# 원본은 유지하고 분석용 데이터에서만
# 구·군 표기와 확인된 좌표 오류를 정제한다.

senior_class_clean = senior_class.copy()

# 구·군명 앞뒤 공백 제거
senior_class_clean["구군명"] = (
    senior_class_clean["구군명"]
    .astype(str)
    .str.strip()
)

# 기존의 잘못된 좌표를 기준으로 해당 시설 1건을 식별한다.
target = (
    senior_class_clean["위도"].between(37.2564, 37.2566)
    & senior_class_clean["경도"].between(127.6483, 127.6485)
)

print("좌표 보정 대상:", target.sum())

display(
    senior_class_clean.loc[
        target,
        ["구군명", "노인교실명", "도로명주소", "위도", "경도"]
    ]
)

# 지도에서 주소 위치를 확인한 좌표로 보정
senior_class_clean.loc[target, "위도"] = 35.2161508578537
senior_class_clean.loc[target, "경도"] = 129.095818075246

print("\n=== 보정 후 ===")
print("위도:", senior_class_clean["위도"].min(),
      "~", senior_class_clean["위도"].max())
print("경도:", senior_class_clean["경도"].min(),
      "~", senior_class_clean["경도"].max())

In [ ]:
# 노인교실 정제 결과를 최종 검증한다.

print("=== 최종 검증 ===")
print("전체 시설:", len(senior_class_clean))
print("구·군 수:", senior_class_clean["구군명"].nunique())

print("\n=== 구·군별 시설 수 ===")
print(senior_class_clean["구군명"].value_counts().sort_index())

print("\n=== 핵심 변수 결측 ===")
print(
    senior_class_clean[
        ["구군명", "노인교실명", "도로명주소", "위도", "경도"]
    ].isna().sum()
)

print("\n=== 중복 ===")
print(
    senior_class_clean.duplicated(
        subset=["노인교실명", "도로명주소"]
    ).sum()
)

print("\n=== 부산 범위 밖 좌표 ===")
outside = ~(
    senior_class_clean["위도"].between(34.9, 35.5)
    & senior_class_clean["경도"].between(128.7, 129.4)
)

print(outside.sum())

## 노인교실 데이터 검증 및 정제 결과

2025년 9월 1일 기준 부산광역시 노인교실 현황 데이터를 확인한 결과
총 132개 시설이 수록되어 있다.

핵심 변수인 시설명, 주소, 구·군, 위도·경도에는 결측값이 없었으며,
시설명과 주소 기준 중복도 확인되지 않았다.

구·군명 검증 과정에서 `영도구 `와 같이 뒤쪽 공백이 포함된 값 1건을 확인하여
앞뒤 공백을 제거하고 `영도구`로 표준화하였다.

공백 제거 후 노인교실 데이터에는 부산 16개 구·군 중 15개 구·군이 포함되어 있으며,
중구에 해당하는 시설 레코드는 원본 데이터에서 확인되지 않았다.
따라서 임의로 시설을 추가하지 않고 원자료의 상태를 그대로 유지한다.

좌표 검증 과정에서는 `행복나눔 노인대학` 1건의 위도·경도가
부산 지역을 벗어난 값으로 입력되어 있음을 확인하였다.
도로명주소 위치를 지도에서 재확인한 후 분석용 데이터의 좌표를 보정하였다.

원본 CSV는 수정하지 않고 보존하며,
위 정제 사항은 분석용 데이터에만 적용하였다.

최종적으로 132개 노인교실 레코드를 고령층 생활서비스 공급 분석에 활용한다.

In [ ]:
# 경로당과 노인교실을 동일한 컬럼 구조로 표준화한다.

senior_center_std = senior_center[
    ["gugun", "name", "address", "lat", "lng"]
].copy()

senior_center_std.columns = [
    "gugun",
    "facility_name",
    "address",
    "lat",
    "lng"
]

senior_center_std["facility_type"] = "경로당"


senior_class_std = senior_class_clean[
    ["구군명", "노인교실명", "도로명주소", "위도", "경도"]
].copy()

senior_class_std.columns = [
    "gugun",
    "facility_name",
    "address",
    "lat",
    "lng"
]

senior_class_std["facility_type"] = "노인교실"


print("경로당:", senior_center_std.shape)
print("노인교실:", senior_class_std.shape)

In [ ]:
# 두 데이터셋에서 동일 주소가 존재하는지 확인한다.
# 같은 장소에서 서로 다른 서비스가 운영되는 경우를 파악하기 위한 검증이다.

cross_overlap = senior_center_std[
    senior_center_std["address"].isin(
        senior_class_std["address"]
    )
]

print("경로당-노인교실 동일 주소 경로당 레코드:", len(cross_overlap))

display(
    cross_overlap[
        ["gugun", "facility_name", "address"]
    ].head(20)
)

In [ ]:
# 경로당과 노인교실을 하나의 고령층 생활서비스 공급 데이터로 통합한다.
# 동일 주소라도 시설유형이 다르면 서로 다른 서비스 공급으로 유지한다.

senior_service = pd.concat(
    [senior_center_std, senior_class_std],
    ignore_index=True
)

print("=== 통합 결과 ===")
print("전체 공급 레코드:", len(senior_service))

print("\n시설유형별")
print(senior_service["facility_type"].value_counts())

print("\n구·군 수:", senior_service["gugun"].nunique())

print("\n구·군별 공급 레코드")
print(
    senior_service.groupby(
        ["gugun", "facility_type"]
    ).size().unstack(fill_value=0)
)

print("\n핵심 변수 결측")
print(
    senior_service[
        ["gugun", "facility_name", "address", "lat", "lng", "facility_type"]
    ].isna().sum()
)

print("\n동일 유형 내 시설명+주소 중복:",
      senior_service.duplicated(
          subset=["facility_type", "facility_name", "address"]
      ).sum())

## 고령층 생활서비스 공급 데이터 구성 결과

고령층 생활서비스 공급 수준을 측정하기 위해
부산광역시 경로당과 노인교실 데이터를 활용하였다.

### 경로당

경로당 데이터는 총 2,374건으로 부산 16개 구·군이 모두 포함되어 있다.
시설명, 주소, 구·군, 좌표에 결측값이 없었으며,
시설명과 주소 기준 중복 및 비정상 좌표도 확인되지 않아 전체 데이터를 활용하였다.

### 노인교실

2025년 9월 1일 기준 노인교실 데이터는 총 132건이다.
구·군명의 불필요한 공백을 제거하였으며,
좌표가 부산 지역을 벗어난 `행복나눔 노인대학` 1건은
도로명주소의 실제 위치를 확인하여 분석용 데이터에서 좌표를 보정하였다.

정제 후 핵심 변수 결측, 시설명·주소 기준 중복,
부산 지역을 벗어난 좌표는 확인되지 않았다.

노인교실 데이터에는 중구에 해당하는 레코드가 없었으며,
원자료에 존재하지 않는 시설을 임의로 추가하지 않았다.

### 데이터 통합

경로당과 노인교실의 컬럼을
`구·군, 시설명, 주소, 위도, 경도, 시설유형`으로 표준화하여 통합하였다.

두 데이터 간 동일 주소 3건이 확인되었으나,
경로당과 노인교실은 서로 다른 유형의 생활서비스이므로
동일 장소에서 함께 운영되는 경우에도 각각의 서비스 공급으로 유지하였다.

최종 고령층 생활서비스 공급 데이터는 다음과 같다.

- 경로당: 2,374건
- 노인교실: 132건
- 총 공급 레코드: 2,506건

본 데이터는 이후 행정동 단위로 공간 집계하여
60대 이상 생활인구 수요와 지역별 서비스 공급 수준을 비교하는 데 활용한다.

In [ ]:
# 최종 고령층 생활서비스 공급 데이터를 processed_data에 저장한다.

senior_processed_path = (
    "/content/drive/MyDrive/부산_BigData_공모전/"
    "02_processed_data/고령층생활서비스_분석용.csv"
)

senior_service.to_csv(
    senior_processed_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 데이터:", senior_service.shape)
print("저장 완료")

## 4. 아동 돌봄서비스 최신 공식자료 보완

아동복지시설 API를 통해 지역아동센터와 다함께돌봄센터 데이터를 1차 수집·검증하였다.

그러나 부산광역시 최신 공식 시설현황과 비교한 결과, 특히 다함께돌봄센터의 경우
API 수록 건수가 최신 현황을 충분히 반영하지 못하는 것으로 확인되었다.

따라서 최종 아동 돌봄서비스 공급 분석에는 부산광역시가 공개한 최신 시설현황을 사용한다.

- 지역아동센터: 2026년 6월 기준 212개소
- 다함께돌봄센터: 2026년 7월 기준 78개소
- 기존 API 자료는 데이터 품질 검증 및 수집 과정 기록으로 보존한다.

두 공식 자료에는 행정동 정보가 포함되어 있으므로, 이를 활용하여 생활인구 데이터와
행정동 단위로 직접 연결한다.

In [ ]:
import os

raw_dir = "/content/drive/MyDrive/부산_BigData_공모전/01_raw_data/공공서비스_공급"

files = os.listdir(raw_dir)

print("공공서비스 공급 원본 파일")
for file in files:
    print("-", file)

In [ ]:
# 공식 아동시설 PDF 경로 확인 및 기본 검증
import os
import glob

pdf_files = glob.glob(os.path.join(raw_dir, "*.pdf"))

print("PDF 파일 수:", len(pdf_files))

for path in pdf_files:
    print("\n파일명:", os.path.basename(path))
    print("파일 크기:", round(os.path.getsize(path) / 1024, 1), "KB")

In [ ]:
# PDF 페이지와 텍스트 추출 가능 여부 확인
!pip -q install pdfplumber

import pdfplumber

for path in pdf_files:
    print("\n" + "=" * 60)
    print("파일:", os.path.basename(path))

    with pdfplumber.open(path) as pdf:
        print("페이지 수:", len(pdf.pages))

        first_page = pdf.pages[0]
        text = first_page.extract_text()

        print("첫 페이지 텍스트 추출:", "성공" if text else "실패")

        if text:
            print("\n[앞부분 미리보기]")
            print(text[:1000])

In [ ]:
# PDF의 표 구조를 페이지별로 추출
import pandas as pd
import pdfplumber
import os

for path in pdf_files:
    print("\n" + "=" * 70)
    print("파일:", os.path.basename(path))

    with pdfplumber.open(path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            tables = page.extract_tables()

            print(f"\n[페이지 {page_num}] 추출된 표 수: {len(tables)}")

            for table_num, table in enumerate(tables, start=1):
                df_temp = pd.DataFrame(table)
                print(
                    f"표 {table_num}: "
                    f"{df_temp.shape[0]}행 × {df_temp.shape[1]}열"
                )
                display(df_temp.head(5))

In [ ]:
# 파일명 유니코드 정규화 후 PDF 전체 표 추출
import unicodedata

def norm_text(text):
    return unicodedata.normalize("NFC", text)

def extract_pdf_tables(path):
    rows = []

    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables()

            for table in tables:
                rows.extend(table)

    return pd.DataFrame(rows)


child_path = [
    p for p in pdf_files
    if "지역아동센터" in norm_text(os.path.basename(p))
][0]

care_path = [
    p for p in pdf_files
    if "다함께돌봄" in norm_text(os.path.basename(p))
][0]

print("지역아동센터 파일:", norm_text(os.path.basename(child_path)))
print("다함께돌봄센터 파일:", norm_text(os.path.basename(care_path)))

child_raw = extract_pdf_tables(child_path)
care_raw = extract_pdf_tables(care_path)

print("\n지역아동센터 추출 크기:", child_raw.shape)
print("다함께돌봄센터 추출 크기:", care_raw.shape)

In [ ]:
# 연번 기준으로 실제 시설 행 구조 검증
def check_row_numbers(df, name):
    first_col = (
        df.iloc[:, 0]
        .astype(str)
        .str.replace("\n", "", regex=False)
        .str.strip()
    )

    numbers = pd.to_numeric(first_col, errors="coerce").dropna().astype(int)

    print(f"\n[{name}]")
    print("전체 추출 행:", len(df))
    print("숫자 연번 행 수:", len(numbers))
    print("최소 연번:", numbers.min())
    print("최대 연번:", numbers.max())
    print("연번 중복 수:", numbers.duplicated().sum())

    expected = set(range(numbers.min(), numbers.max() + 1))
    actual = set(numbers)

    print("누락 연번:", sorted(expected - actual))
    print(
        "중복 연번:",
        sorted(numbers[numbers.duplicated(keep=False)].unique().tolist())
    )

    # 연번이 아닌 행 확인
    non_number = df[pd.to_numeric(first_col, errors="coerce").isna()]
    print("\n연번이 아닌 행:")
    display(non_number)


check_row_numbers(child_raw, "지역아동센터")
check_row_numbers(care_raw, "다함께돌봄센터")

In [ ]:
# PDF에서 분리된 연속행을 앞 시설 행에 복원
care_fixed = care_raw.copy()

first_col = (
    care_fixed.iloc[:, 0]
    .astype(str)
    .str.replace("\n", "", regex=False)
    .str.strip()
)

continuation_idx = care_fixed[
    pd.to_numeric(first_col, errors="coerce").isna()
].index.tolist()

# 헤더 행 제외
continuation_idx = [
    idx for idx in continuation_idx
    if str(care_fixed.loc[idx, 0]).strip() != "연번"
]

print("분리된 연속행 인덱스:", continuation_idx)

for idx in continuation_idx:
    prev_idx = idx - 1

    print(f"\n[복원 전] 앞 행 index={prev_idx}")
    display(care_fixed.loc[[prev_idx]])

    print(f"[연속 행] index={idx}")
    display(care_fixed.loc[[idx]])

    # 연속행의 값은 앞 행의 빈 칸에만 채움
    for col in care_fixed.columns:
        prev_val = care_fixed.loc[prev_idx, col]
        cont_val = care_fixed.loc[idx, col]

        prev_empty = pd.isna(prev_val) or str(prev_val).strip() in ["", "None"]
        cont_exists = not pd.isna(cont_val) and str(cont_val).strip() not in ["", "None"]

        if prev_empty and cont_exists:
            care_fixed.loc[prev_idx, col] = cont_val

# 복원한 연속행 제거
care_fixed = care_fixed.drop(index=continuation_idx).reset_index(drop=True)

print("\n복원 후 다함께돌봄센터 크기:", care_fixed.shape)

In [ ]:
# 연번이 있는 실제 시설 행만 추출
columns = [
    "연번", "구군명", "법정동", "행정동", "기관명", "주소",
    "연락처", "운영요일", "휴무일",
    "학기시작", "학기종료", "방학시작", "방학종료",
    "이용대상", "이용요금"
]

def make_facility_table(df):
    result = df.copy()
    result.columns = columns

    result["연번"] = pd.to_numeric(result["연번"], errors="coerce")
    result = result[result["연번"].notna()].copy()

    result["연번"] = result["연번"].astype(int)

    return result.reset_index(drop=True)


child_center = make_facility_table(child_raw)
together_care = make_facility_table(care_fixed)

print("지역아동센터:", child_center.shape)
print("다함께돌봄센터:", together_care.shape)

In [ ]:
# 핵심 분석 변수 품질 검증
key_cols = ["구군명", "법정동", "행정동", "기관명", "주소"]

def validate_facility(df, name):
    print("\n" + "=" * 60)
    print(name)
    print("시설 수:", len(df))
    print("구군 수:", df["구군명"].nunique())

    print("\n[핵심 변수 결측]")
    print(df[key_cols].isna().sum())

    print("\n[빈 문자열]")
    for col in key_cols:
        blank = df[col].astype(str).str.strip().eq("").sum()
        print(f"{col}: {blank}")

    print("\n[기관명+주소 중복]")
    dup = df.duplicated(subset=["기관명", "주소"], keep=False)
    print("중복 행 수:", dup.sum())

    if dup.sum() > 0:
        display(
            df.loc[dup, ["연번", "구군명", "행정동", "기관명", "주소"]]
            .sort_values(["기관명", "주소"])
        )

    print("\n[구군별 시설 수]")
    display(
        df.groupby("구군명")
        .size()
        .reset_index(name="시설수")
    )


validate_facility(child_center, "지역아동센터")
validate_facility(together_care, "다함께돌봄센터")

In [ ]:
# 분석에 필요한 핵심 변수만 구성
child_center_std = child_center[
    ["구군명", "법정동", "행정동", "기관명", "주소"]
].copy()

together_care_std = together_care[
    ["구군명", "법정동", "행정동", "기관명", "주소"]
].copy()

child_center_std["시설유형"] = "지역아동센터"
together_care_std["시설유형"] = "다함께돌봄센터"

child_service = pd.concat(
    [child_center_std, together_care_std],
    ignore_index=True
)

child_service = child_service.rename(columns={
    "구군명": "gugun",
    "법정동": "legal_dong",
    "행정동": "admin_dong",
    "기관명": "facility_name",
    "주소": "address",
    "시설유형": "facility_type"
})

print("통합 데이터 크기:", child_service.shape)
print("\n시설유형별 건수:")
print(child_service["facility_type"].value_counts())

print("\n구군 수:", child_service["gugun"].nunique())

display(child_service.head())

In [ ]:
# 통합 후 최종 품질 검증
print("[핵심 변수 결측]")
print(child_service.isna().sum())

print("\n[완전 중복]")
print(child_service.duplicated().sum())

print("\n[동일 유형 내 기관명+주소 중복]")
same_type_dup = child_service.duplicated(
    subset=["facility_type", "facility_name", "address"],
    keep=False
)
print(same_type_dup.sum())

print("\n[구군 × 시설유형 공급 현황]")
child_district_summary = (
    child_service
    .groupby(["gugun", "facility_type"])
    .size()
    .unstack(fill_value=0)
)

child_district_summary["전체"] = child_district_summary.sum(axis=1)

display(child_district_summary)
print("\n전체 시설 수:", child_district_summary["전체"].sum())

In [ ]:
# 아동 돌봄서비스 분석용 데이터 저장
processed_dir = "/content/drive/MyDrive/부산_BigData_공모전/02_processed_data"

output_path = os.path.join(
    processed_dir,
    "아동돌봄서비스_공식현황_분석용.csv"
)

child_service.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path)
print("저장 행 수:", len(child_service))

In [ ]:
# PDF 셀 내부 줄바꿈 및 불필요한 공백 정리
text_cols = [
    "gugun",
    "legal_dong",
    "admin_dong",
    "facility_name",
    "address"
]

for col in text_cols:
    child_service[col] = (
        child_service[col]
        .astype(str)
        .str.replace("\n", "", regex=False)
        .str.strip()
    )

print("줄바꿈 정리 후 확인:")
display(child_service.head())

print("\n컬럼별 줄바꿈 잔존 개수:")
for col in text_cols:
    print(col, child_service[col].str.contains("\n", regex=False).sum())

In [ ]:
output_path = os.path.join(
    processed_dir,
    "아동돌봄서비스_공식현황_분석용.csv"
)

child_service.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path)
print("저장 행 수:", len(child_service))

### 아동 돌봄서비스 공식 현황 전처리 결과

기존 아동복지시설 API를 검토한 결과, 지역아동센터는 비교적 최신 현황과 유사했으나
다함께돌봄센터는 API 수록 건수가 최신 공식 현황을 충분히 반영하지 못하는 것으로 확인되었다.

이에 최종 공급 분석에는 부산광역시의 최신 공식 시설현황을 사용하였다.

- 지역아동센터: 2026년 6월 기준 212개소
- 다함께돌봄센터: 2026년 7월 기준 78개소
- 총 아동 돌봄서비스 공급: 290개 레코드
- 두 시설유형 모두 부산 16개 구·군에 분포
- 구군명, 법정동, 행정동, 기관명, 주소의 핵심 변수 결측 없음
- 동일 시설유형 내 기관명+주소 중복 없음

PDF 표 추출 과정에서 다함께돌봄센터 1개 시설의 부가정보가 별도 행으로 분리되어 추출되었으나,
연번 1~78의 연속성 및 중복 여부를 검증하여 실제 시설 수가 78개임을 확인하였다.
분석에는 운영시간 등의 부가정보를 사용하지 않고 공식 자료에 포함된 구군명, 법정동,
행정동, 기관명, 주소와 시설유형만 활용하였다.

최종 분석용 데이터는 290개 서비스 공급 레코드로 구성하였다.
공식 자료에 행정동 정보가 포함되어 있으므로 아동 돌봄서비스는 별도의 좌표 기반 공간결합 없이
구군명과 행정동을 기준으로 생활인구 데이터와 연결한다.

In [ ]:
# 생활인구 행정동 데이터 불러오기
population_path = os.path.join(
    processed_dir,
    "생활인구_행정동별_202503_202512.csv"
)

population = pd.read_csv(population_path, encoding="utf-8-sig")

print("생활인구 데이터 크기:", population.shape)
print("컬럼:", population.columns.tolist())
display(population.head())

In [ ]:
# 구군 + 행정동 기준 매칭키 생성
child_service["dong_key"] = (
    child_service["gugun"].str.strip()
    + " "
    + child_service["admin_dong"].str.strip()
)

population["dong_key"] = (
    population["행정동명"]
    .astype(str)
    .str.replace("\n", "", regex=False)
    .str.strip()
)

population_keys = set(population["dong_key"])
child_keys = set(child_service["dong_key"])

matched = child_keys & population_keys
unmatched = child_keys - population_keys

print("아동시설이 존재하는 행정동 수:", len(child_keys))
print("생활인구와 매칭된 행정동 수:", len(matched))
print("매칭되지 않은 행정동 수:", len(unmatched))

print("\n[매칭되지 않은 행정동]")
for dong in sorted(unmatched):
    print("-", dong)

In [ ]:
# 시설 레코드 기준 매칭 여부 검증
child_service["population_match"] = (
    child_service["dong_key"].isin(population_keys)
)

print("전체 시설:", len(child_service))
print("매칭 시설:", child_service["population_match"].sum())
print("미매칭 시설:", (~child_service["population_match"]).sum())

if (~child_service["population_match"]).sum() > 0:
    display(
        child_service.loc[
            ~child_service["population_match"],
            [
                "gugun",
                "legal_dong",
                "admin_dong",
                "facility_name",
                "address",
                "facility_type"
            ]
        ]
    )

In [ ]:
# 생활인구 데이터에서 신호동 및 해당 행정동코드 확인
display(
    population[
        population["행정동명"].astype(str).str.contains("신호", na=False)
        | (population["행정동코드"].astype(str) == "2644059000")
    ]
)

print("\n강서구 행정동 목록:")
display(
    population[
        population["행정동명"].astype(str).str.startswith("강서구 ")
    ][["행정동코드", "행정동명"]]
)

In [ ]:
# 신호동 행정동명 표준화
mask = population["행정동코드"].astype(str) == "2644059000"

print("수정 대상 행 수:", mask.sum())
print("수정 전:", population.loc[mask, "행정동명"].tolist())

population.loc[mask, "행정동명"] = "강서구 신호동"

population["dong_key"] = (
    population["행정동명"]
    .astype(str)
    .str.replace("\n", "", regex=False)
    .str.strip()
)

print("수정 후:", population.loc[mask, "행정동명"].tolist())

In [ ]:
# 보정 후 아동시설-생활인구 매칭 재검증
population_keys = set(population["dong_key"])

child_service["population_match"] = (
    child_service["dong_key"].isin(population_keys)
)

print("전체 시설:", len(child_service))
print("매칭 시설:", child_service["population_match"].sum())
print("미매칭 시설:", (~child_service["population_match"]).sum())

unmatched_dongs = sorted(
    set(child_service.loc[
        ~child_service["population_match"],
        "dong_key"
    ])
)

print("미매칭 행정동:", unmatched_dongs)

In [ ]:
# 보정된 생활인구 데이터 재저장
population_save = population.drop(
    columns=["dong_key"],
    errors="ignore"
)

population_save.to_csv(
    population_path,
    index=False,
    encoding="utf-8-sig"
)

print("생활인구 파일 보정 저장 완료")
print("행 수:", len(population_save))

In [ ]:
# 행정동별 시설유형 공급량 집계
child_supply_dong = (
    child_service
    .groupby(["dong_key", "facility_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# 시설이 없는 행정동도 포함하도록 206개 생활인구 기준으로 결합
child_supply_dong = (
    population[["행정동코드", "행정동명", "dong_key"]]
    .merge(
        child_supply_dong,
        on="dong_key",
        how="left"
    )
)

facility_cols = ["지역아동센터", "다함께돌봄센터"]

child_supply_dong[facility_cols] = (
    child_supply_dong[facility_cols]
    .fillna(0)
    .astype(int)
)

child_supply_dong["아동돌봄시설_전체"] = (
    child_supply_dong["지역아동센터"]
    + child_supply_dong["다함께돌봄센터"]
)

print("행정동 수:", len(child_supply_dong))
print("지역아동센터 합계:", child_supply_dong["지역아동센터"].sum())
print("다함께돌봄센터 합계:", child_supply_dong["다함께돌봄센터"].sum())
print("전체 공급 합계:", child_supply_dong["아동돌봄시설_전체"].sum())

display(child_supply_dong.head())

In [ ]:
# 행정동별 공급 분포 확인
print("[아동돌봄시설 전체 공급 분포]")
print(
    child_supply_dong["아동돌봄시설_전체"]
    .value_counts()
    .sort_index()
)

print("\n시설이 없는 행정동 수:",
      (child_supply_dong["아동돌봄시설_전체"] == 0).sum())

print("\n공급 시설 수 상위 10개 행정동:")
display(
    child_supply_dong[
        ["행정동명", "지역아동센터", "다함께돌봄센터", "아동돌봄시설_전체"]
    ]
    .sort_values("아동돌봄시설_전체", ascending=False)
    .head(10)
)

In [ ]:
# 행정동별 아동돌봄 공급 데이터 저장
child_supply_path = os.path.join(
    processed_dir,
    "아동돌봄서비스_행정동별_공급.csv"
)

child_supply_dong.to_csv(
    child_supply_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", child_supply_path)
print("저장 행 수:", len(child_supply_dong))

### 행정동별 아동 돌봄서비스 공급 집계 결과

공식 시설현황의 행정동 정보를 생활인구 데이터의 206개 행정동 체계와 매칭하였다.

- 지역아동센터: 212개소
- 다함께돌봄센터: 78개소
- 전체 아동 돌봄서비스: 290개소
- 생활인구 행정동과 시설 290개 전부 매칭 완료
- 206개 행정동 중 아동 돌봄시설이 존재하는 행정동: 151개
- 공급시설이 없는 행정동: 55개
- 행정동별 최대 공급: 기장군 정관읍 8개소

시설이 없는 행정동을 곧바로 서비스 부족지역으로 판단하지 않는다.
후속 분석에서 20대 미만 생활인구의 절대 규모와 구성비를 함께 고려하여
지역별 아동 돌봄수요 대비 공급 수준을 비교한다.

행정동별 지역아동센터 수, 다함께돌봄센터 수 및 전체 아동돌봄시설 수를
후속 수요-공급 불균형 분석의 공급지표로 활용한다.

In [ ]:
!pip -q install admdongkor

import admdongkor
import geopandas as gpd

print("admdongkor 설치 완료")

In [ ]:
# admdongkor 사용 가능한 함수와 데이터 구조 확인
import admdongkor

print("모듈 위치:", admdongkor.__file__)
print("\n사용 가능한 항목:")
print([x for x in dir(admdongkor) if not x.startswith("_")])

In [ ]:
# 제공 버전과 get 함수 사용법 확인
import admdongkor
import inspect

print("[현재 데이터 버전]")
print(admdongkor.data_version)

print("\n[사용 가능한 버전]")
print(admdongkor.versions())

print("\n[get 함수 사용법]")
print(inspect.signature(admdongkor.get))
print(inspect.getdoc(admdongkor.get))

In [ ]:
# 2025년 말 행정동 경계 불러오기
busan_boundary = admdongkor.get(
    "20251231",
    level="emd",
    detail=True,
    crs="EPSG:4326"
)

print("전체 경계:", busan_boundary.shape)
print("CRS:", busan_boundary.crs)
print("컬럼:", busan_boundary.columns.tolist())

display(busan_boundary.head())

In [ ]:
# 부산 경계 추출 및 생활인구 206개 행정동과 코드 대조
busan_boundary = busan_boundary[
    busan_boundary["sidonm"] == "부산광역시"
].copy()

boundary_codes = set(busan_boundary["emdcd"].astype(str))
population_codes = set(population["행정동코드"].astype(str))

only_boundary = boundary_codes - population_codes
only_population = population_codes - boundary_codes

print("[부산 행정동 경계]")
print("행정동 수:", len(busan_boundary))
print("구·군 수:", busan_boundary["sggnm"].nunique())
print("행정동 코드 중복:", busan_boundary["emdcd"].duplicated().sum())

print("\n[신호동 확인]")
display(
    busan_boundary.loc[
        busan_boundary["emdnm"].str.contains("신호", na=False),
        ["emdcd", "emdnm", "sggnm", "sidonm"]
    ]
)

print("\n[생활인구와 코드 비교]")
print("생활인구 행정동 수:", len(population))
print("경계 코드 수:", len(boundary_codes))
print("생활인구 코드 수:", len(population_codes))
print("경계에만 있는 코드:", len(only_boundary))
print("생활인구에만 있는 코드:", len(only_population))

if only_boundary:
    print("\n경계에만 존재:")
    display(
        busan_boundary[
            busan_boundary["emdcd"].astype(str).isin(only_boundary)
        ][["emdcd", "sggnm", "emdnm"]]
    )

if only_population:
    print("\n생활인구에만 존재:")
    display(
        population[
            population["행정동코드"].astype(str).isin(only_population)
        ][["행정동코드", "행정동명"]]
    )

In [ ]:
# 고령층 시설 좌표를 공간데이터로 변환
senior_gdf = gpd.GeoDataFrame(
    senior_service.copy(),
    geometry=gpd.points_from_xy(
        senior_service["lng"],
        senior_service["lat"]
    ),
    crs="EPSG:4326"
)

print("고령층 서비스:", senior_gdf.shape)
print("CRS:", senior_gdf.crs)
print("\n시설유형별:")
print(senior_gdf["facility_type"].value_counts())

In [ ]:
# 행정동 경계를 GeoDataFrame으로 정리
busan_boundary_gdf = busan_boundary.copy()

busan_boundary_gdf = gpd.GeoDataFrame(
    busan_boundary_gdf,
    geometry="geom",
    crs="EPSG:4326"
)

boundary_for_join = busan_boundary_gdf[
    ["emdcd", "emdnm", "sggnm", "geom"]
].copy()

print("경계 행정동:", len(boundary_for_join))
print("CRS:", boundary_for_join.crs)

In [ ]:
# 시설 좌표를 행정동 경계에 공간결합
senior_matched = gpd.sjoin(
    senior_gdf,
    boundary_for_join,
    how="left",
    predicate="within"
)

print("공간결합 후 행 수:", len(senior_matched))
print("매칭 성공:", senior_matched["emdcd"].notna().sum())
print("매칭 실패:", senior_matched["emdcd"].isna().sum())
print("중복 매칭으로 증가한 행:", len(senior_matched) - len(senior_gdf))

In [ ]:
# 행정동 코드 자료형 통일 후 고령층 서비스 공급 집계
population_merge = population[
    ["행정동코드", "행정동명"]
].copy()

population_merge["행정동코드"] = (
    population_merge["행정동코드"].astype(str)
)

senior_matched_merge = senior_matched.copy()
senior_matched_merge["emdcd"] = (
    senior_matched_merge["emdcd"].astype(str)
)

# 행정동별 시설유형 공급 집계
senior_supply_dong = (
    senior_matched_merge
    .groupby(["emdcd", "facility_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# 생활인구 206개 행정동을 기준으로 결합
senior_supply_dong = population_merge.merge(
    senior_supply_dong,
    left_on="행정동코드",
    right_on="emdcd",
    how="left"
)

facility_cols = ["경로당", "노인교실"]

senior_supply_dong[facility_cols] = (
    senior_supply_dong[facility_cols]
    .fillna(0)
    .astype(int)
)

senior_supply_dong["고령층생활서비스_전체"] = (
    senior_supply_dong["경로당"]
    + senior_supply_dong["노인교실"]
)

senior_supply_dong = senior_supply_dong.drop(columns="emdcd")

# 최종 검증
print("행정동 수:", len(senior_supply_dong))
print("행정동코드 중복:", senior_supply_dong["행정동코드"].duplicated().sum())
print("경로당 합계:", senior_supply_dong["경로당"].sum())
print("노인교실 합계:", senior_supply_dong["노인교실"].sum())
print("전체 공급 합계:", senior_supply_dong["고령층생활서비스_전체"].sum())

display(senior_supply_dong.head())

In [ ]:
# 행정동별 공급 분포 확인
print("[고령층 생활서비스 전체 공급 분포]")
print(
    senior_supply_dong["고령층생활서비스_전체"]
    .value_counts()
    .sort_index()
)

zero_count = (
    senior_supply_dong["고령층생활서비스_전체"] == 0
).sum()

print("\n시설이 없는 행정동 수:", zero_count)

print("\n공급 시설 수 상위 10개 행정동:")
display(
    senior_supply_dong[
        ["행정동명", "경로당", "노인교실", "고령층생활서비스_전체"]
    ]
    .sort_values("고령층생활서비스_전체", ascending=False)
    .head(10)
)

In [ ]:
# 행정동별 고령층 공급 데이터 저장
senior_output_path = (
    "/content/drive/MyDrive/부산_BigData_공모전/"
    "02_processed_data/고령층생활서비스_행정동별_공급.csv"
)

senior_supply_dong.to_csv(
    senior_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", senior_output_path)
print("저장 shape:", senior_supply_dong.shape)

### 고령층 생활서비스 행정동 공간매칭 및 공급 집계 결과

경로당 및 노인교실의 위·경도 좌표를 2025년 말 부산 행정동 경계와 공간결합하여
생활인구 데이터와 동일한 206개 행정동 체계로 변환하였다.

- 부산 행정동 경계: 206개
- 생활인구 행정동: 206개
- 행정동 코드 전체 일치
- 강서구 신호동 분동 반영 확인
- 경로당: 2,374개
- 노인교실: 132개
- 전체 고령층 생활서비스 공급 레코드: 2,506개
- 공간매칭 성공: 2,506개
- 공간매칭 실패: 0개
- 중복 공간매칭: 0개
- 행정동별 집계 후 경로당·노인교실 합계 보존 확인
- 고령층 생활서비스가 없는 행정동: 1개

행정동별 공급량은 큰 편차를 보였으며,
기장읍 87개, 정관읍 82개 등 일부 읍·동에서 높은 공급량이 확인되었다.

다만 시설 수 자체는 지역의 면적, 생활인구 규모 및 인구구성과 관계없이
서비스 충분·부족 여부를 의미하지 않는다.
따라서 본 결과는 행정동별 서비스 공급 현황으로 활용하고,
후속 분석에서 60대 이상 생활인구의 규모 및 구성비 등 지역수요와 결합한다.

경로당과 노인교실은 서로 다른 서비스 유형이므로 개별 공급량을 유지하며,
두 유형의 단순 합계인 `고령층생활서비스_전체`는 보조적인 공급 규모 지표로 활용한다.